# M2 Notebook 18 — Calibration and Decision Thresholds

**Status:** Runnable first edition

## Learning objectives

- Measure probability calibration.
- Apply Platt calibration.
- Choose thresholds using decision costs.

In [ ]:
from srai_math.utils import environment_info,set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()
from srai_ml import (
    LogisticRegressionGD,PlattCalibrator,brier_score,calibration_curve,
    expected_cost,optimal_threshold,train_test_split,
)


In [ ]:
rng=np.random.default_rng(18)
X=rng.normal(size=(800,3))
logit=-1+1.5*X[:,0]-.7*X[:,1]+.4*X[:,2]
true_p=1/(1+np.exp(-logit))
y=rng.binomial(1,true_p)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.4,seed=18)
model=LogisticRegressionGD(.1,3000).fit(Xtr,ytr)
raw=model.predict_proba(Xte)[:,1]
distorted=np.clip(raw**1.8,0,1)
cal=PlattCalibrator(.1,2500).fit(np.log((distorted+1e-6)/(1-distorted+1e-6)),yte)
calibrated=cal.predict_proba(np.log((distorted+1e-6)/(1-distorted+1e-6)))
{"distorted_brier":brier_score(yte,distorted),
 "calibrated_brier":brier_score(yte,calibrated)}


## Reliability diagram

In [ ]:
mp1,fp1,c1=calibration_curve(yte,distorted,10)
mp2,fp2,c2=calibration_curve(yte,calibrated,10)
fig,ax=plt.subplots(figsize=(6,6))
ax.plot([0,1],[0,1],linestyle="--")
ax.plot(mp1,fp1,marker="o",label="uncalibrated")
ax.plot(mp2,fp2,marker="o",label="calibrated")
ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed frequency")
ax.legend(); ax.set_title("Calibration Curves")
plt.show()


## Cost-sensitive thresholding

In [ ]:
threshold,cost=optimal_threshold(yte,calibrated,
                                   false_positive_cost=1,
                                   false_negative_cost=5)
{"optimal_threshold":threshold,
 "expected_cost":cost,
 "cost_at_0_5":expected_cost(yte,calibrated,.5,1,5)}


## Decision Intelligence case

Calibrated probabilities enable explicit cost-based and capacity-based decision thresholds.

## Key insight

A good ranking model may still produce poor probabilities; calibration and threshold selection are separate tasks.